# 5.17 eval 就會停止計算梯度嗎？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**model.eval()後loss還能backward嗎？

eval改變dropout等模組行爲，不停用autograd。no_grad停用這次運算的圖。requires_grad=False只凍結特定參數。三者各解決不同問題。

**把直覺寫成數學：**能否backward由計算圖與requires_grad決定，不由model.training一個旗標決定。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
layer = nn.Linear(2, 1)
x = torch.ones(1, 2)
layer.eval()
output = layer(x)
print("eval的grad_fn", output.grad_fn)
output.sum().backward()
with torch.no_grad():
    other = layer(x)
print("no_grad的grad_fn", other.grad_fn)
assert output.requires_grad and not other.requires_grad

**如何讀結果：**教師/reference通常要同時eval、requires_grad_(False)、no_grad；理由分開看。

**只改一件事：**只freeze權重但讓輸入requires_grad=True，觀察仍能回傳到輸入。
